# Python Applications for Regression and Prediction

In this section, we will learn how to build, evaluate, and interpret basic regression and predictive models using Python. We will cover key steps in the modeling process, including selecting features, fitting models to data, and assessing model performance. Throughout this notebook, you will have the opportunity to apply concepts hands-on with code examples and visualizations, helping you gain practical experience with real-world datasets.

**By the end of this section, you should be able to:**

- Understand the basic workflow of building predictive models
- Implement simple regression and classification models in Python
- Evaluate and interpret model results using appropriate metrics and visualizations
- Appreciate the importance of model selection, validation, and interpretation in data-driven decision-making

Let us get started with modeling!

---

## Common Quantitative Analysis Models and Python Applications

1. **Linear Regression**  
   Linear regression is a linear approach to modeling the relationship between a dependent variable and one or more independent variables. The dependent variable is continuous, while the independent variables can be continuous or categorical. The goal is to find the best-fitting line through the data, minimizing the sum of squared differences between observed and predicted values.

2. **Logistic Regression**  
   Logistic regression is a statistical model used to predict the probability of a binary outcome based on one or more independent variables. The dependent variable is binary, and the independent variables can be continuous or categorical. It uses the logistic function to model probabilities and is commonly used for classification.


# Case Study: Analyzing Canadian Provincial Weather Data

In the second part of our workshop, we will conduct a case study using a real-world weather dataset that includes provincial-level weather information for Canada. The data contains columns such as province name, region, yearly temperature, and precipitation, allowing us to explore various aspects of weather patterns across different provinces and years.

## Dataset Properties

- The CSV holds one temperature and one precipitation record per province; the 2010 and 2011 series are built in this notebook
- Features include temperature, precipitation, and region for each province
- Data includes both original and engineered features, such as snow indicators and temperature differences between years

## Objectives

With this dataset, our goals are to:

- Explore and understand the structure of the weather data
- Clean and organize the data (rename columns, set appropriate indices, select relevant features)
- Create new features to compare weather conditions across years
- Analyze which provinces and years are more likely to experience snow
- Visualize relationships between temperature and precipitation for Canadian provinces
- Practice basic data analysis skills such as filtering, sorting, and plotting in Python using pandas and matplotlib

## Analysis Steps

To achieve these objectives, we will:

1. Load the weather dataset and inspect its shape and contents
2. Clean the data by selecting important columns and renaming them for clarity
3. Create new columns, including year-over-year temperature difference and estimated precipitation
4. Engineer new features such as binary indicators for possible snow events based on weather conditions
5. Filter and sort data to answer specific analysis questions about provinces and years
6. Visualize the relationship between temperature and precipitation, labeling points by province
7. Summarize our findings about provincial weather patterns and snow likelihood

---

## Reading the import block

Python starts with a small core language. Anything beyond that lives in a *library* (also called a package or a module) that you have to load before you can use it. `import` is the statement that loads one.

`import pandas as pd` loads the library named `pandas` and gives it the short local name `pd`. From that point on, `pd.read_csv(...)` means "call the `read_csv` function that lives inside pandas". The `as pd` part is only a nickname. `import pandas` alone would also work, but then you would type `pandas.read_csv(...)` every single time. `pd`, `np`, `plt` and `sm` are the nicknames the whole Python community uses, so keep them.

What each of the four does in this notebook:

- `pandas` (`pd`): tables. Reading the CSV, filtering rows, adding columns, sorting.
- `matplotlib.pyplot` (`plt`): static charts. `plt.scatter`, `plt.bar`, `plt.show`.
- `numpy` (`np`): numeric arrays and math. Used here for `np.polyfit` and `np.sort`.
- `statsmodels.api` (`sm`): statistics. Used here for `sm.add_constant` and `sm.OLS`.

`matplotlib.pyplot` has a dot in it: `matplotlib` is the package and `pyplot` is a module inside it. The dot means "look inside".

Gotcha: you only run this cell once per session, but every name it defines disappears when the kernel restarts. If a cell suddenly reports `NameError: name 'pd' is not defined`, come back and re-run this cell before debugging anything else.

Two more imports appear further down, inside the cells that use plotly (`import plotly.graph_objects as go`). Importing part-way through a notebook works fine. The usual convention is to put all imports at the top; the author kept plotly next to its first use so the introduction stays local.

In [ ]:
import pandas as pd              # tables: read_csv, DataFrame, Series, filtering
import matplotlib.pyplot as plt  # static charts: scatter, bar, xlabel, show
import numpy as np               # numeric arrays and math: polyfit, sort
import statsmodels.api as sm     # statistics: add_constant, OLS regression

# Basic Operations

### Loading the file, and what `read_csv` gives back

`pd.read_csv('../../../data/province_weather.csv')` reads a comma-separated text file off disk and returns a **DataFrame**.

The string is a *relative path*. It is resolved from the folder this notebook runs in, which is `python/Day 2/Part-2`. Each `../` means "go up one folder": up to `Day 2`, up to `python`, up to the repository root, then down into `data/`. Move the notebook and this path breaks with `FileNotFoundError`. That is the most common first-day error in the whole bootcamp.

`df.info()` prints a structural summary rather than the data itself. Read the printed block as:

- `RangeIndex: 13 entries, 0 to 12` - the table has 13 rows and their labels are the integers 0 through 12.
- `Data columns (total 4 columns)` - four columns.
- `Non-Null Count` - how many rows hold a real value in that column. All four report `13 non-null`, so nothing is missing.
- `Dtype` - how the column is stored. `object` means text (`Shortnam`, `Region`). `float64` means a 64-bit decimal number (`Temperature`, `Precipitation`).

`info()` returns nothing. It prints. So it is written as a statement on its own line and never assigned to a variable.

Alternatives for the same job. `df.shape` gives `(13, 4)` and nothing else. `df.dtypes` gives only the type of each column. `df.describe()` gives count, mean, standard deviation, minimum, quartiles and maximum for the numeric columns only. Use `info()` when you want to know what is *there* and `describe()` when you want to know what the numbers *look like*.

In [ ]:
# read_csv parses the text file into a DataFrame. The path is relative to this
# notebook's own folder: three ../ steps up to the repo root, then into data/.
df = pd.read_csv('../../../data/province_weather.csv')

# info() prints structure, not data: row count, column names, how many non-null
# values each column holds, and each column's storage type. It returns None.
df.info()

### A DataFrame is a table. A Series is one column of it.

These are the only two pandas objects you need today.

A **DataFrame** is a rectangular table with three parts:

1. **columns** - named, and each column holds one kind of value.
2. **an index** - the row labels down the left edge. Here they are `0, 1, 2, ..., 12`, created automatically because the CSV had no ID column of its own.
3. **the values** - the actual cells.

A **Series** is a single column pulled out of a DataFrame: one sequence of values plus that same index. `df['Temperature']` is a Series of 13 floats labeled 0 through 12.

How this differs from what you already know:

- A Python **list** such as `[-6.0, -1.0, -18.0]` has positions only. You reach item 2 with `x[2]`, and you cannot ask for "the temperature of the row named NL".
- A Python **dict** such as `{'NL': -6.0, 'PEI': -1.0}` has labels but no column arithmetic and no tabular alignment.
- A **Series** has both labels and positions, and it does whole-column arithmetic. `df['Temperature'] * 2` doubles all 13 values at once. Compare `[-6.0, -1.0] * 2` on a plain list, which *repeats the list* and gives four items. That difference catches people constantly.

Displaying `df` on its own line prints the whole table, all 13 rows. `df.head()` prints the first 5. `head` takes an argument, so `df.head(3)` gives 3 rows, and `df.tail()` gives the last 5. With 13 rows you could print everything, but `head()` is the habit to build: on a two-million-row file, printing everything will freeze your notebook.

One Jupyter rule explains why the next two cells are separate: a cell auto-displays only the value of its **last** line. Two displays need two cells, or an explicit `print()` on each.

In [ ]:
df   # the last line of a cell auto-displays its value; here, the full 13-row table

In [ ]:
df.head()   # first 5 rows only. df.head(3) for three, df.tail() for the last five.

### Looking at the pieces directly

Run the next cell to see the parts of the DataFrame separately. Nothing is modified.

The last two lines make the most important syntax distinction in pandas: **one name in the brackets gives a Series, a list of names gives a DataFrame**, even when the list holds only one name.

In [ ]:
print(df.shape)                 # (rows, columns) -> (13, 4)
print(list(df.columns))         # the four column names
print(list(df.index))           # the row labels: 0 through 12
print(type(df))                 # DataFrame: the whole table
print(type(df['Temperature']))  # Series: one column

# One string in the brackets -> Series (1-D).
# A LIST of strings -> DataFrame (2-D), even with a single name in the list.
print(type(df['Temperature']), type(df[['Temperature']]))

## Q1: Why are we creating a new column called `snow` using temperature and precipitation data?

We want to identify and flag the weather conditions in which snow is likely to occur. By combining temperature and precipitation, we can create a simple indicator: if the temperature is low and precipitation is present, then snow is more likely. The new `snow` column helps us easily spot and analyze these cases in our data.


### How the next cell works

Three ideas, stacked.

**1. `df.copy()` makes an independent duplicate.** `weather = df.copy()` gives you a second table that shares no memory with the first. If you wrote `weather = df` instead, both names would point at the *same* table, and `weather['snow'] = ...` would quietly add a `snow` column to `df` as well. Copying keeps the raw loaded data clean so you can always go back to it. This notebook does `weather = df.copy()` several more times to reset.

**2. Column arithmetic is element-wise.** `weather['Temperature'] * weather['Precipitation']` multiplies row 0 by row 0, row 1 by row 1, and so on, returning a new Series of 13 numbers. No loop required. For row 0 that is `-6.0 * 71.0 = -426.0`.

**3. Comparing a Series to a number gives a boolean Series.** `(...) < -10` tests each of the 13 products against -10 and returns 13 `True`/`False` values. That is what gets stored, so the new `snow` column has dtype `bool`.

Assigning to a column name that does not exist creates it. Assigning to one that does exist overwrites it. There is no separate "add column" function.

On this data the rule flags 11 of the 13 provinces. The two `False` rows are Quebec and Ontario, both sitting at exactly 0.0 C: `0.0 * 5.0 = 0.0` and `0.0 * 59.0 = 0.0`, and neither is below -10.

Gotcha: the rule is arbitrary, not meteorology. Nova Scotia at -18.0 C with only 2.0 mm of precipitation still passes, because `-18 * 2 = -36`. Treat `snow` as a teaching feature, not a forecast.

Alternative formulation, easier to defend because each condition is separately meaningful:

```python
weather['snow'] = (weather['Temperature'] < 0) & (weather['Precipitation'] > 5)
```

That is a genuinely different rule: it flags 8 provinces instead of 11, disagreeing on NS, NB and SK. The cell below keeps the product form because Q9 reuses the same formula for 2010 and 2011.

In [ ]:
weather = df.copy()   # independent duplicate; edits here never touch df

# Temperature * Precipitation multiplies the two columns row by row (13 products),
# then < -10 compares each product to -10 and returns 13 True/False values.
# Assigning that boolean Series to a new name creates the column.
weather['snow'] = (weather['Temperature'] * weather['Precipitation']) < -10

weather.head()   # 11 of the 13 rows come out True; QC and ON are False (both 0.0 C)

## Q2: How can we view the weather data sorted in alphabetical order by region?

Use code to display the top rows of the dataset after sorting it by the "Region" column in ascending (A-Z) order.


### `sort_values`, and where the row labels go

`weather.sort_values(by='Region', ascending=True)` reorders the rows.

- `by='Region'` names the column to sort on. Text sorts alphabetically, numbers numerically.
- `ascending=True` means A to Z, or low to high. `ascending=False` reverses it. `True` is the default, so it could be omitted; spelling it out is clearer.
- The return value is a **new** DataFrame. `weather` itself is untouched, which is exactly why displaying `weather` again straight afterwards still shows the original order. To keep the sorted version you must assign it: `weather = weather.sort_values(by='Region')`.

The detail beginners miss: **the index labels travel with the rows**. After sorting, the leftmost column reads `8, 9, 6, 3, 0` instead of `0, 1, 2, 3, 4`. Row 8 is Alberta, first alphabetically. A label is a permanent name for a row, not its current position. Use `.reset_index(drop=True)` if you want fresh 0-based labels after sorting.

You can sort on several columns at once: `sort_values(by=['Region', 'Temperature'])` sorts by Region and breaks ties with Temperature.

Alternatives. `weather.sort_index()` sorts by the row labels rather than by a column. `weather.nlargest(3, 'Temperature')` pulls the top 3 by a numeric column in one step, and is the cleaner way to do what a later cell does with `sort_values(...).head(3)`.

In [ ]:
# Returns a NEW sorted DataFrame; weather is unchanged because nothing is assigned.
# Watch the index column in the output: it reads 8, 9, 6, 3, 0 - the row labels
# follow their rows through the sort. Alberta (label 8) is first alphabetically.
weather.sort_values(by='Region', ascending=True).head()

In [ ]:
weather   # proof that sort_values did not modify anything: original order, labels 0-12

## Q3: How can we display only the weather data for the region "Alberta"?

Write code to filter the dataset so that it shows only the rows where the Region is Alberta.


### Boolean masks, and why `weather[weather['Region'] == 'Alberta']` works

Read it from the inside out.

**Step 1.** `weather['Region'] == 'Alberta'` compares all 13 region names against the text `'Alberta'`. The result is a Series of 13 `True`/`False` values, exactly one of which is `True` (row 8). That object is called a **boolean mask**.

**Step 2.** `weather[ mask ]` hands the mask back to the DataFrame. Square brackets containing a boolean Series mean "keep the rows where the mask is `True` and drop the rest". Result: a one-row DataFrame.

So `weather[...]` has two completely different jobs, decided by what is inside the brackets:

- `weather['Region']` - brackets hold a **string**, so you get that column back as a Series.
- `weather[mask]` - brackets hold a **boolean Series**, so you get filtered rows back as a DataFrame.

**`=` versus `==`.** `=` assigns. `==` asks a question. `weather['Region'] = 'Alberta'` would overwrite the entire Region column with the word Alberta and destroy the data. `weather['Region'] == 'Alberta'` asks which rows match and changes nothing. A missing second `=` is the most expensive typo in this notebook.

The comparison is exact and case-sensitive. `'alberta'`, or `'Alberta '` with a trailing space, match zero rows and return an **empty DataFrame, not an error**. When a filter returns nothing, check spelling and stray whitespace before suspecting anything else.

Alternatives that do the same thing:

- `weather.loc[weather['Region'] == 'Alberta']` - explicit label-based selection. You need `.loc` if you also want to pick columns in the same call: `weather.loc[weather['Region'] == 'Alberta', ['Region', 'Temperature']]`.
- `weather.query("Region == 'Alberta'")` - a string expression, with column names written bare. Shorter to read. Slower on small tables, and column names containing spaces need backticks.
- `weather[weather['Region'].isin(['Alberta', 'Quebec'])]` - for matching several values at once instead of chaining `|` over and over.

In [ ]:
# Inside the brackets: weather['Region'] == 'Alberta' produces 13 True/False values.
# Outside: passing that boolean Series to weather[...] keeps only the True rows.
# Exactly one province matches, so the result is a 1-row DataFrame (label 8).
weather_filtered = weather[weather['Region'] == 'Alberta']
weather_filtered.head(5)   # asks for 5 rows but only 1 exists, so 1 is shown

## Q4: How can we view only the weather data where precipitation is greater than 10 and temperature is above -10?

Write code to filter the dataset for rows where precipitation is more than 10 and temperature is greater than -10.


### Combining conditions: `&`, `|`, and those parentheses

Each condition on its own produces a mask. To combine two masks you need element-wise operators:

- `&` is AND: `True` only where **both** masks are `True`.
- `|` is OR: `True` where **either** is `True`.
- `~` is NOT: it flips every value.

**Why not the words `and` and `or`?** Python's `and` and `or` expect one single true-or-false answer, not 13 of them. Handing them a Series raises `ValueError: The truth value of a Series is ambiguous`. `&` and `|` work position by position and give 13 answers back, which is what filtering needs.

**Why every condition needs its own parentheses.** In Python, `&` binds *more tightly* than `>`. Written without parentheses, `weather['Precipitation'] > 10 & weather['Temperature'] > -10` is parsed as `weather['Precipitation'] > (10 & weather['Temperature']) > -10`, which is not what you meant and will error. Wrapping each comparison, `(A > 10) & (B > -10)`, forces the comparisons to run first. Parenthesize every time; there is no case where it hurts.

The first cell below uses `&`: precipitation above 10 **and** temperature above -10. Four provinces survive - NL, PEI, ON and AB. `.head()` would show up to 5, so you are seeing the complete result.

The second cell uses `|`: precipitation above 10 **or** temperature above -10. A looser test, so more rows pass: 10 of the 13. Only NS, NB and NT fail both conditions. `.head()` shows the first 5 of those 10.

Gotcha to carry forward: a filtered result keeps the original index labels (`0, 1, 5, 8` in the `&` case) and is a slice of the parent table. Assigning a new column straight onto it can trigger a `SettingWithCopyWarning`, covered at Q7 below.

In [ ]:
# Two masks joined with & (AND). Each comparison gets its own parentheses because
# & binds tighter than >, so without them Python would group the expression wrongly.
# Result: 4 provinces satisfy both conditions - NL, PEI, ON, AB.
weather_filtered = weather[(weather['Precipitation'] > 10) & (weather['Temperature'] > -10)]
weather_filtered.head()

In [ ]:
# Same two conditions joined with | (OR), so a row needs to satisfy only one of them.
# A looser test lets more rows through: 10 of 13 pass. Only NS, NB and NT fail both.
weather_filtered2 = weather[(weather['Precipitation'] > 10) | (weather['Temperature'] > -10)]
weather_filtered2.head()   # first 5 of the 10 matching rows

## Q5: How can we add a new column to the weather data that shows temperature in Fahrenheit instead of Celsius?

Write code to create a function to convert Celsius to Fahrenheit, and then apply it to the "Temperature" column to create a new column called "Temperature_F".


### Defining a function and applying it to a column

`def c_to_f(c):` defines a function named `c_to_f` taking one input called `c`. `return c * 9/5 + 32` sends a value back to whoever called it. On its own, `c_to_f(-6.0)` returns `21.2`.

`weather['Temperature'].apply(c_to_f)` runs that function once for each value in the Temperature Series and collects the 13 results into a new Series. Notice that `c_to_f` is passed **without parentheses**. `apply(c_to_f)` hands over the function itself. `apply(c_to_f())` would try to call it with no argument and raise `TypeError`.

Assigning the result creates `Temperature_F`. Row 0: -6.0 C becomes 21.2 F. Row 2: -18.0 C becomes -0.4 F.

Alternatives, in the order you should reach for them:

1. **Vectorized arithmetic**: `weather['Temperature_F'] = weather['Temperature'] * 9/5 + 32`. Identical result, no function needed, and much faster, because pandas does the whole column in one compiled operation instead of 13 separate Python calls. On 13 rows the difference is invisible; on a million rows `apply` can be tens of times slower. Prefer this whenever the operation is plain arithmetic.
2. **`.apply(lambda c: c * 9/5 + 32)`**: the same as the cell below without naming the function. Fine for a one-off.
3. **A named function plus `.apply`**: what the cell below does. Worth it when the logic is long, has branches, or gets reused. `c_to_f` is reused later in the Task section, so naming it pays off here.

Floating point footnote. `-18.0 * 9/5 + 32` is stored as `-0.3999999999999986` and pandas rounds it to `-0.4` for display. Binary floats cannot represent every decimal exactly. Never test a float with `==`; use `round()` or `np.isclose()`.

In [ ]:
# A plain Python function: takes one number in Celsius, returns it in Fahrenheit.
def c_to_f(c):
    return c * 9/5 + 32

# .apply runs c_to_f once per value in the column and gathers the 13 results.
# c_to_f is passed WITHOUT parentheses - apply calls it for you.
# Vectorized equivalent, faster and preferred for plain arithmetic:
#   weather['Temperature_F'] = weather['Temperature'] * 9/5 + 32
weather['Temperature_F'] = weather['Temperature'].apply(c_to_f)
weather.head()

## Q6: How can we compare temperature values between two different years and calculate the difference for each row?

Write code to add two new columns to the weather data for temperatures in 2010 and 2011, then create a third column that shows the difference between 2011 and 2010 temperatures.


### Assigning a Python list as a column

`temp_2010` and `temp_2011` are ordinary Python lists holding 13 integers each. They are invented numbers for the exercise; the CSV contains only one temperature column.

`weather['Temperature_2010'] = temp_2010` turns a list into a column, matching **by position**: list item 0 goes to the row labeled 0, item 1 to row 1, and so on. The list length must equal the number of rows, otherwise pandas raises `ValueError: Length of values (12) does not match length of index (13)`.

Because the values are Python `int`, the two new columns come out as dtype `int64`, unlike `Temperature` which is `float64`.

`weather['Temperature_2011'] - weather['Temperature_2010']` subtracts row by row and produces 13 differences. Row 0: `-5 - (-2) = -3`. A negative difference means 2011 was colder than 2010 in that province.

Gotcha, and it matters: assignment by position is blind. Pandas does not check that item 0 really belongs to Newfoundland. If the list came from a source sorted differently, you would silently attach the wrong numbers to the wrong provinces and every later result would be wrong with no error message. When both sides carry meaningful labels, build a Series with a matching index and let pandas align, or use `merge` on a key column.

In [ ]:
# Plain Python lists of 13 invented values, one per province, in row order.
temp_2010 = [-2, -14, -6, -19, -8, -11, -3, -17, -9, -4, -13, -5, -10]
temp_2011 = [-5, -10, -15, -3, -18, -7, -12, -1, -20, -8, -6, -9, -11]

# Assigning a list creates a column, matched BY POSITION. The list length must
# equal the row count (13) or pandas raises ValueError. These are ints, so the
# new columns are dtype int64 while Temperature stays float64.
weather['Temperature_2010'] = temp_2010
weather['Temperature_2011'] = temp_2011

# Element-wise subtraction of two columns: 13 differences. Negative means 2011
# was colder than 2010. Row 0: -5 - (-2) = -3.
weather['Temperature_Diff'] = weather['Temperature_2011'] - weather['Temperature_2010']
weather.head()

### A branching function, then undoing it

`temp_indicator` takes one number and returns the text `"High"` or `"Low"` depending on whether it exceeds 5. The parameter is named `F` because the function is applied to the Fahrenheit column. Row 0 (21.2 F) gives `"High"`, row 2 (-0.4 F) gives `"Low"`.

This is the case where `.apply` genuinely earns its place: the logic contains an `if`, so plain column arithmetic cannot express it.

Vectorized alternative for a two-way choice:

```python
weather['temp_indicator'] = np.where(weather['Temperature_F'] > 5, 'High', 'Low')
```

`np.where(condition, value_if_true, value_if_false)` produces the same 13 labels in one step. For more than two buckets, use `pd.cut` for numeric bins or `np.select` for a list of conditions.

The cell after that throws the column away again with `weather.drop(columns=['temp_indicator'])`. Points worth knowing:

- `columns=[...]` takes a **list** of names, even when dropping just one.
- `drop` returns a new DataFrame. `weather = weather.drop(...)` is what makes the removal stick.
- `weather.drop(columns=['temp_indicator'], inplace=True)` modifies in place instead of returning. Both work; the reassignment form is preferred now, and `inplace=True` is being discouraged in modern pandas.
- The older spelling is `weather.drop(['temp_indicator'], axis=1)`, where `axis=1` means columns and `axis=0` means rows. `columns=` is clearer and harder to get backwards.
- Dropping a name that is not present raises `KeyError`. Pass `errors='ignore'` to let it pass silently.

The cell after the drop re-displays the first three rows so you can confirm the column is gone. Verifying after a destructive step is a good habit.

In [ ]:
#Create a new col to define whether the temp is high(greater than 5) or low (less than 5)
# The logic has an if, so it cannot be written as plain column arithmetic.
# .apply is the right tool here. Vectorized alternative for a two-way split:
#   weather['temp_indicator'] = np.where(weather['Temperature_F'] > 5, 'High', 'Low')
def temp_indicator(F):
    if F>5:
        return "High"
    else:
        return "Low"

weather['temp_indicator']= weather['Temperature_F'].apply(temp_indicator)
weather.head(3)

In [ ]:
# drop returns a NEW DataFrame without those columns, so the result must be
# reassigned for the change to stick. columns= always takes a list, even for one
# name. Older equivalent: weather.drop(['temp_indicator'], axis=1).
weather = weather.drop(columns=['temp_indicator'])
weather.head(3)

In [ ]:
weather.head(3)   # confirm the drop worked: temp_indicator is gone

### Seeing a mask by itself

Q3 and Q4 hid the boolean mask inside the square brackets. Here it is on its own, so you can look at the object directly. Nothing in the next cell modifies `weather`.

Two things to take away. Summing a boolean Series counts the `True` values, because `True` behaves as 1 and `False` as 0, so `mask.sum()` is a row counter and `mask.mean()` is the matching proportion. And `weather[mask]` returns a smaller table with the same number of columns: filtering removes rows, never columns.

In [ ]:
# A mask is just a Series of True/False, one entry per row, sharing the same index.
mask = weather['Temperature'] > -10
print(mask)

# True counts as 1 and False as 0, so summing a mask counts the matching rows.
print('rows above -10 C:', mask.sum())          # 6

# Feeding the mask back into square brackets keeps only the True rows.
# Filtering removes rows, never columns: 6 rows, still 9 columns.
print('shape after filtering:', weather[mask].shape)

## Q7: How can we keep only selected columns in the weather data and rename them for clarity?

Write code to select only the columns "Shortnam", "Region", "Temperature_2010", "Temperature_2011", and "Precipitation". Then rename "Shortnam" to "Province" and "Precipitation" to "Precipitation_2010".


### Selecting columns, renaming them, and the double brackets

The first cell below is only a look at the table before the change. The second one does two separate things.

**Selecting a subset of columns.**

```python
weather = weather[['Shortnam', 'Region', 'Temperature_2010', 'Temperature_2011', 'Precipitation']]
```

The **outer** brackets are the selection operator. The **inner** brackets are a Python list of column names. Together they mean "give me a DataFrame containing exactly these five columns, in this order". This is where the famous double-bracket confusion comes from:

- `weather['Region']` - one string, returns a **Series**.
- `weather[['Region']]` - a list holding one name, returns a **DataFrame** with one column.

The order of the list is the order of the output columns, so this filters and reorders at the same time. Everything not listed (`Temperature`, `snow`, `Temperature_F`, `Temperature_Diff`) is discarded. `weather` is overwritten, so those columns are gone from this point on and later cells cannot use them.

Alternatives: `weather.drop(columns=[...])` when you want to remove a few and keep many; `weather.loc[:, [...]]` when you want rows and columns in one call; `weather.filter(items=[...])` for pattern-based selection.

**Renaming.**

```python
weather = weather.rename(columns={'Shortnam': 'Province', 'Precipitation': 'Precipitation_2010'})
```

`columns=` takes a **dictionary** shaped `{old_name: new_name}`. Names not mentioned are left alone. `rename` returns a new DataFrame, so once again the result has to be reassigned.

`Shortnam` (a typo carried in from the source CSV) becomes `Province`. `Precipitation` becomes `Precipitation_2010`, which bakes in the assumption that the single precipitation column represents 2010. Q8 builds directly on that assumption.

Alternative: `weather.columns = ['Province', 'Region', ...]` replaces all names at once by position. Shorter but fragile, because you must supply exactly as many names as there are columns and in the right order. Prefer the dictionary.

**`SettingWithCopyWarning`, because you will hit it.** Write `subset = weather[weather['Temperature'] > -10]` and then `subset['flag'] = 1`, and pandas cannot always tell whether `subset` is an independent table or a view into `weather`, so it prints:

```
SettingWithCopyWarning: A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead
```

It is a warning, not an error. The code still runs, but the assignment may not land where you expect. The fix is one word: `subset = weather[weather['Temperature'] > -10].copy()`. That is precisely why the Q1 cell and the plotting cells start with `df.copy()`. The next cell does not trigger the warning even though it slices columns, because `.rename()` on the following line returns a brand-new object that no longer references the parent.

In [ ]:
weather.head(3)   # look at the table before the columns are cut down

In [ ]:
# Outer brackets = selection. Inner brackets = a Python list of column names.
# A list of names returns a DataFrame with exactly those columns, in that order.
# Everything else (Temperature, snow, Temperature_F, Temperature_Diff) is dropped.
weather = weather[['Shortnam', 'Region', 'Temperature_2010', 'Temperature_2011', 'Precipitation']]

# rename takes a dict of {old_name: new_name}. Unlisted names are left alone,
# and rename returns a NEW DataFrame, so the result must be reassigned.
weather = weather.rename(columns={'Shortnam': 'Province', 'Precipitation': 'Precipitation_2010'})
weather.head()

### Missing values: `NaN`, `isna`, `dropna`, `fillna`

`NaN` stands for "Not a Number". Pandas uses it as the marker for a missing value: a blank field in the CSV, a failed type conversion, or a row with no match in a merge. It is not zero and it is not the empty string. Any arithmetic touching `NaN` produces `NaN`, and `NaN == NaN` evaluates to `False`, so you can never test for it with `==`.

`weather.isna()` returns a DataFrame of the same shape holding `True` wherever a value is missing. Chaining `.sum()` collapses each column to a count, because `True` sums as 1. The printed result here is `0` for all five columns: this dataset is complete. Running the check anyway is the right habit - you check *before* you trust.

`weather.dropna()` removes every row containing at least one `NaN`. With no missing values, it returns all 13 rows unchanged. Note that the cell does **not** assign the result, so `weather` is not modified; the output is only displayed. Harmless here because nothing would be dropped, but if you actually meant to clean the table you would need `weather = weather.dropna()`.

Useful variations:

- `weather.dropna(subset=['Temperature_2010'])` - drop a row only when that one column is missing.
- `weather.dropna(axis=1)` - drop whole *columns* that contain any missing value.
- `weather.dropna(how='all')` - drop only rows where *every* value is missing.

**`dropna` versus `fillna`.** `dropna` deletes, `fillna` substitutes. `weather['Precipitation_2010'].fillna(0)` puts 0 in the gaps; `fillna(weather['Precipitation_2010'].mean())` puts the column mean in. The choice is a modeling decision, not a formatting one. Dropping rows shrinks the sample and biases it whenever missingness is related to the outcome. Filling with a mean keeps the row count but shrinks the variance and understates standard errors. For an economics assignment, state which you did and why.

In [ ]:
# isna() gives a same-shaped table of True/False for "is this value missing".
# .sum() then counts the True values down each column, since True sums as 1.
# Every column reports 0 here, so this dataset has no missing values.
weather.isna().sum()

In [ ]:
# dropna() removes any row holding at least one NaN. Nothing is missing, so all
# 13 rows come back. The result is NOT assigned, so weather itself is unchanged;
# to actually clean the table you would write weather = weather.dropna().
weather.dropna()

## Q8: How can we create a new column to estimate precipitation for 2011 by increasing the 2010 precipitation by 10 percent?

Write code to add a new column called "Precipitation_2011" that is 1.1 times the value of "Precipitation_2010".


### Multiplying a column by a number

`weather['Precipitation_2010'] * 1.1` multiplies every value by 1.1. Combining a single number with a Series applies it to all 13 elements; this is called *broadcasting*. Row 0: `71.0 * 1.1 = 78.1`. Row 2: `2.0 * 1.1 = 2.2`.

The result is a float Series, and assigning it under a new name creates the `Precipitation_2011` column.

Be clear about what this is: a made-up assumption that 2011 precipitation was exactly 10 percent above 2010 in every province. It is not measured data. It exists so Q9 has a second year to work with. Anything you conclude from `Precipitation_2011` is a statement about the assumption, not about Canadian weather.

`* 1.1` is a 10 percent increase because 1 + 0.10 = 1.1. For a 10 percent decrease you would multiply by 0.9, not subtract 1.1. To compute a growth *rate* from two real columns you would use `(new - old) / old`.

In [ ]:
# A single number combined with a Series is applied to all 13 values (broadcasting).
# 1.1 means a 10 percent increase, since 1 + 0.10 = 1.1. Row 0: 71.0 * 1.1 = 78.1.
# This is an assumption invented for the exercise, not measured 2011 data.
weather['Precipitation_2011'] = weather['Precipitation_2010'] * 1.1
weather.head()

## Q9: How can we identify possible snow conditions for the years 2010 and 2011 using temperature and precipitation data?

Write code to create two new columns, "snow_2010" and "snow_2011", that use the rule (temperature times precipitation less than -10) to flag likely snow events in each year.


### Reapplying the snow rule to each year

The same rule as Q1, `temperature * precipitation < -10`, applied twice with the year-specific columns. Each line produces a boolean Series and stores it as a new column.

Results on this data:

- `snow_2010` is `True` for all 13 provinces. Every 2010 temperature is negative and every 2010 precipitation is positive, and no product lands in the narrow band between -10 and 0.
- `snow_2011` is `True` for 12 and `False` for one: Saskatchewan, where `-1 * 3.3 = -3.3`, which is not below -10.

A feature that is `True` everywhere carries no information. `snow_2010` would be worthless as a predictor in any model, because it cannot distinguish one province from another. Checking `weather['snow_2010'].value_counts()` before using a flag costs two seconds and saves a lot of confusion.

The comparison operators available are `<`, `>`, `<=`, `>=`, `==`, `!=`. On a boolean column, `.sum()` counts the `True` values and `.mean()` gives the proportion.

In [ ]:
# Same rule as Q1, now applied per year. Each line builds a boolean Series and
# stores it as a new column. snow_2010 comes out True for all 13 provinces;
# snow_2011 is True for 12 and False for SK, where -1 * 3.3 = -3.3 (not < -10).
weather['snow_2010'] = (weather['Temperature_2010'] * weather['Precipitation_2010']) < -10
weather['snow_2011'] = (weather['Temperature_2011'] * weather['Precipitation_2011']) < -10
weather.head()

# Visualizing Data

Three different plotting toolkits appear below, and the notebook draws several of the same charts twice - once with matplotlib and once with plotly. That is deliberate, but it means you need to know which one you are reading.

1. **matplotlib** (`plt.scatter`, `plt.bar`, `plt.show`). The base library. You build a picture by issuing commands one at a time against a hidden "current figure", then call `plt.show()` to render it. The output is a static image. Anything starting with `plt.` is matplotlib.
2. **pandas `.plot()`** (`top_3.plot(kind='line')` further down). A thin convenience wrapper that calls matplotlib for you, using the DataFrame's own index for the x axis and its column names for the legend. Shortest code for a quick look, least control.
3. **plotly** (`go.Figure`, `fig.add_trace`, `fig.show`). Produces an interactive HTML chart with hover tooltips, zoom, pan and click-to-hide series. You build an object describing the chart and then show it, rather than issuing drawing commands.

Which to pick:

| Situation | Use |
| --- | --- |
| Quick look while exploring | `df.plot()` |
| Figure for a paper or a PDF | matplotlib |
| Chart in a notebook or webpage that people will hover over | plotly |
| Interactive chart with minimal code | plotly express (`px.scatter`) |

Plotly express is worth knowing before you read the long blocks below. `px.scatter(weather, x='Temperature', y='Precipitation', text='Shortnam')` reproduces most of the first plotly cell in a single line, because `px` fills in sensible defaults for everything the long form spells out. Read the long `graph_objects` form once to understand the pattern, then use `px` for your own work until you need a detail it will not give you.

## Q10: How can we create a scatter plot to visualize the relationship between temperature and precipitation for each province, and label each point with the province name?

Write code to make a scatter plot of "Temperature" versus "Precipitation" and label each point with the corresponding province using the "Shortnam" column.


### The matplotlib scatter, line by line

**Read the first line carefully: `weather = df.copy()` throws away everything built in Q1 through Q9.** From here on `weather` holds only the four original CSV columns again - `Shortnam`, `Region`, `Temperature`, `Precipitation`. The `snow`, `Temperature_F`, `Temperature_2010`, `Temperature_2011` and `Precipitation_2011` columns no longer exist. If you skip around and later hit `KeyError: 'Temperature_2010'`, this reset is why.

Then, in order:

- `plt.scatter(x, y)` draws one dot per pair. Two Series of 13 values in, 13 dots out. The first argument is always x, the second always y.
- `for i in range(len(weather)):` loops `i` over `0, 1, 2, ..., 12`. `len(weather)` is the row count, 13.
- `plt.text(x, y, s)` writes the string `s` at the data coordinates `(x, y)`. Called 13 times, so each dot gets its province code next to it.
- `plt.xlabel`, `plt.ylabel` and `plt.title` add the axis labels and the title. Always put units on an axis label: `Temperature (C)` is far more useful than `Temperature`.
- `plt.show()` renders the accumulated figure and clears the canvas. Without it, the next plotting cell would draw on top of this one.

**The indexing gotcha inside the loop.** `weather['Temperature'][i]` looks like "the i-th value" but it is not. On a Series, `[i]` looks up the **label** `i`, not position `i`. It works here only because `df.copy()` still carries the default labels `0..12`, so label and position happen to coincide. Sort or filter first and it breaks silently:

```python
srt = weather.sort_values('Temperature')
srt['Temperature'][0]        # -6.0  -> the row LABELED 0, now sitting in the middle
srt['Temperature'].iloc[0]   # -20.0 -> the actual first row
```

`.iloc[i]` is always positional and `.loc[label]` is always label-based. Use them and the ambiguity disappears. A loop that never has this problem:

```python
for _, row in weather.iterrows():
    plt.text(row['Temperature'], row['Precipitation'], row['Shortnam'])
```

In [ ]:
weather = df.copy()   # RESET: back to the four original columns. Everything added
                      # in Q1-Q9 (snow, Temperature_F, the year columns) is gone.

# Plot the data
# One dot per province: 13 (x, y) pairs. First argument is x, second is y.
plt.scatter(weather['Temperature'], weather['Precipitation'])

# Label each dot. range(len(weather)) counts 0, 1, ..., 12.
# plt.text(x, y, s) writes the string s at those data coordinates.
# Note: [i] looks up the LABEL i, not position i. It works only because the
# index is still 0..12 here. After a sort or filter, use .iloc[i] instead.
for i in range(len(weather)):
    plt.text(weather['Temperature'][i], weather['Precipitation'][i], weather['Shortnam'][i])

plt.xlabel('Temperature (C)')      # always include units on an axis label
plt.ylabel('Precipitation (mm)')
plt.title('Temperature vs Precipitation')
plt.show()                         # render the figure and clear the canvas

### Reading the plotly block

The next cell is long, but it contains only four ideas. Learn them here and every later plotly cell reads quickly.

**1. `import plotly.graph_objects as go`.** `graph_objects` is plotly's explicit, low-level interface. Every chart type is a class inside it: `go.Scatter`, `go.Bar`, `go.Figure`.

**2. A `Figure` is a container.** `fig = go.Figure()` creates an empty chart holding no data. Nothing is drawn yet. Contrast matplotlib, where `plt.scatter(...)` draws immediately. Plotly builds a full description of the chart first and renders it once at the end.

**3. A `trace` is one dataset drawn on the figure.** A trace is a set of points, bars or lines together with the styling for them. `fig.add_trace(go.Scatter(...))` builds one scatter trace and attaches it. Add a second trace and both appear on the same axes with a legend - which is exactly how the regression cell later puts a fitted line on top of the data points.

**4. Styling is passed as nested `dict(...)` calls.** `marker=dict(size=12, color=...)` is a Python dictionary describing the markers. Plotly's options are grouped by topic, so dictionaries inside dictionaries are normal. `dict(size=12)` and `{'size': 12}` mean the same thing; plotly's own documentation uses the `dict(...)` spelling.

Now the arguments, in the order they appear.

Inside `go.Scatter(...)`:

- `x=weather['Temperature']`, `y=weather['Precipitation']` - the 13 coordinates.
- `mode='markers+text'` - what to draw at each point. The pieces are `'markers'`, `'lines'` and `'text'`, joined with `+`. So this means dots plus a label. A later cell uses `'lines+markers+text'`.
- `text=weather['Shortnam']` - the label for each point, and also what `%{text}` refers to in the tooltip.
- `textposition='top center'` - where the label sits relative to its dot.
- `marker=dict(...)` - the dot styling:
  - `size=12` - dot diameter in pixels.
  - `color=weather['Temperature']` - passing a *column* rather than a single color name makes the color vary with the value, so cold and mild provinces look different.
  - `colorscale='Viridis'` - the palette mapping numbers to colors. Viridis is perceptually uniform and survives grayscale printing, which makes it a good default.
  - `colorbar=dict(title='Temperature (C)')` - draws the color legend strip on the right and titles it.
  - `line=dict(width=1, color='black')` - a one-pixel black outline **around each dot**. This `line` sits inside `marker`, so it is the dot border, not a connecting line.
  - `opacity=0.8` - slight transparency so overlapping dots remain visible.
  - `symbol='circle'` - marker shape.
- `name='Locations'` - the trace's name in the legend.
- `hovertemplate=...` - the tooltip text. `%{text}`, `%{x}` and `%{y}` are placeholders filled from that point's data. `<b>` and `<br>` are HTML for bold and line break. The `+` signs just join the three string pieces into one.

Inside `fig.update_layout(...)`, which controls everything outside the data:

- `title`, `xaxis_title`, `yaxis_title` - the text.
- `width=800`, `height=600` - figure size in pixels.
- `plot_bgcolor='white'` - background of the plotting area, replacing plotly's default light gray.
- `xaxis=dict(...)` and `yaxis=dict(...)` - per-axis styling. `showline` draws the axis line, `linewidth` and `linecolor` style it, `mirror=True` repeats it on the opposite side to close the box, `gridcolor` sets the gridline color, `zeroline=False` removes the heavy line drawn at zero.
- `font=dict(size=16)` - base font size for the whole figure.
- `margin=dict(l=60, r=40, t=80, b=60)` - padding in pixels on the left, right, top and bottom. Increase `l` when the y-axis numbers get cut off.

`fig.update_traces(...)` then edits traces that already exist, reapplying the marker outline and setting the label font. Here it partly repeats what `go.Scatter` already set, which is harmless but redundant.

`fig.show()` renders it. Nothing appears on screen before that line runs.

**The one-line version.** `px.scatter(weather, x='Temperature', y='Precipitation', text='Shortnam', color='Temperature')` produces nearly the same chart, because plotly express supplies defaults for everything spelled out below.

In [ ]:
import plotly.graph_objects as go   # graph_objects = plotly's explicit interface

weather = df.copy()   # reset again to the four original columns

# Create scatter trace with text labels
fig = go.Figure()   # an EMPTY figure. Nothing is drawn until traces are added.

# A trace is one dataset plus its styling. add_trace attaches it to the figure.
fig.add_trace(go.Scatter(
    x=weather['Temperature'],       # 13 x coordinates
    y=weather['Precipitation'],     # 13 y coordinates
    mode='markers+text',            # draw dots AND text labels ('lines' also available)
    text=weather['Shortnam'],       # the label per point; also feeds %{text} below
    textposition='top center',      # where the label sits relative to its dot
    marker=dict(                    # nested dict: everything about the dots
        size=12,                    # diameter in pixels
        color=weather['Temperature'],  # a COLUMN, so color varies with the value
        colorscale='Viridis',       # palette mapping numbers to colors
        colorbar=dict(title='Temperature (C)'),  # the color legend strip on the right
        line=dict(width=1, color='black'),       # dot OUTLINE (line inside marker)
        opacity=0.8,                # slight transparency so overlaps stay visible
        symbol='circle'             # marker shape
    ),
    name='Locations',               # legend entry for this trace
    hovertemplate=                  # tooltip text; %{...} are filled per point,
        "<b>%{text}</b><br>" +      # <b> is bold and <br> is a line break (HTML)
        "Temperature: %{x} C<br>" +
        "Precipitation: %{y} mm"
))

# update_layout controls everything OUTSIDE the data: titles, size, axes, fonts.
fig.update_layout(
    title='Temperature vs Precipitation',
    xaxis_title='Temperature (C)',
    yaxis_title='Precipitation (mm)',
    width=800,                      # figure size in pixels
    height=600,
    plot_bgcolor='white',           # plotting area background (default is gray)
    xaxis=dict(
        showline=True,              # draw the axis line
        linewidth=2,
        linecolor='black',
        mirror=True,                # repeat it on the opposite side to close the box
        gridcolor='lightgrey',
        zeroline=False,             # drop the heavy line at zero
    ),
    yaxis=dict(
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True,
        gridcolor='lightgrey',
        zeroline=False,
    ),
    font=dict(size=16),             # base font size for the whole figure
    margin=dict(l=60, r=40, t=80, b=60)   # padding in px: left, right, top, bottom
)

# update_traces edits traces that already exist. Here it re-sets the dot outline
# (redundant, already set above) and sets the label font.
fig.update_traces(
    marker=dict(
        line=dict(width=1, color='black')
    ),
    textfont=dict(
        size=12,
        color='black'
    )
)

fig.show()   # nothing is rendered until this line runs

# Task

1. Add the temperature data from 2010 and 2011 to the data set, convert the temperature data to Fahrenheit and visualize the temperature change.

2. For the 3 warmest provinces in 2010, visualize the temperature over the years.

3. Run a regression analysis on the temperature data against precipitation data for the years 2010 and 2011.

### Rebuild, convert, and compare the two years

`weather = df.copy()` resets to the four original columns yet again.

`weather['temp_2010'] = temp_2010` reuses the lists defined back in Q6. Those variables are still in memory even though the columns they created were discarded. Note the new column names are lowercase `temp_2010` and `temp_2011`, different from the `Temperature_2010` used earlier. Same numbers, different names.

`.apply(c_to_f)` then **overwrites** each column with its Fahrenheit version, reusing the function from Q5. After those two lines the columns hold Fahrenheit, not Celsius: -2 C becomes 28.4 F.

`weather['temp_diff'] = weather['temp_2011'] - weather['temp_2010']` gives the change from 2010 to 2011, in Fahrenheit degrees. Row 0: `23.0 - 28.4 = -5.4`.

A unit subtlety worth stating out loud: a *difference* of 1 degree C equals a difference of 1.8 degrees F, and the +32 offset cancels when you subtract. So `temp_diff` here is exactly 1.8 times the Celsius difference computed in Q6 - the -3 C difference for NL becomes -5.4 F. Converting first and subtracting second is correct as long as the axis is labeled in F, which the code does.

`plt.bar(x, height)` draws one bar per category. The first argument is the category labels (`Shortnam`), the second is the bar heights (`temp_diff`). Bars below zero point downwards, which is what you want for a change measure.

One-line alternative: `weather.plot(kind='bar', x='Shortnam', y='temp_diff')` produces the same chart through pandas and adds the legend automatically.

In [ ]:
weather = df.copy()             # reset to the four original CSV columns
weather['temp_2010'] = temp_2010   # reuse the lists defined back in Q6
weather['temp_2011'] = temp_2011   # note: lowercase names this time

# OVERWRITE each column with its Fahrenheit version, reusing c_to_f from Q5.
# After these two lines the columns hold F, not C: -2 C becomes 28.4 F.
weather['temp_2010'] = weather['temp_2010'].apply(c_to_f)
weather['temp_2011'] = weather['temp_2011'].apply(c_to_f)

# Change from 2010 to 2011, in Fahrenheit degrees. Row 0: 23.0 - 28.4 = -5.4.
# A difference in F is 1.8x the same difference in C (the +32 offset cancels).
weather['temp_diff'] = weather['temp_2011'] - weather['temp_2010']

# Each province's temperature difference
# plt.bar(categories, heights): one bar per province. Negative bars point down.
# Pandas one-liner alternative: weather.plot(kind='bar', x='Shortnam', y='temp_diff')
plt.bar(weather['Shortnam'], weather['temp_diff'])
plt.xlabel('Province')
plt.ylabel('Temperature Difference (F)')
plt.title('Temperature Difference by Province')
plt.show()

### The same bar chart in plotly

The structure is identical to the scatter above: an empty `go.Figure()`, one `add_trace`, one `update_layout`, then `fig.show()`. Only the trace class changes, from `go.Scatter` to `go.Bar`.

What is new:

- `go.Bar(x=..., y=...)` - `x` is the category and `y` is the bar height. With `go.Scatter` both axes are numeric coordinates; with `go.Bar` the x axis is categorical.
- `marker=dict(color=weather['temp_diff'], colorscale='RdBu', ...)` - color by the very quantity being plotted. `'RdBu'` is a *diverging* palette running red - white - blue, which suits a measure with a meaningful zero: provinces that cooled and provinces that warmed get visually opposite colors. Use a diverging scale for changes and differences, and a sequential scale like `'Viridis'` for magnitudes.
- `text=weather['temp_diff'].round(1)` - the number printed on each bar. `.round(1)` keeps one decimal so labels do not read `-5.400000000000002`.
- `textposition='auto'` - plotly places the label inside or outside the bar depending on available space. The alternatives are `'inside'`, `'outside'` and `'none'`.
- `hovertemplate="<b>%{x}</b><br>Temperature Diff: %{y} F<extra></extra>"` - `<extra></extra>` suppresses the gray side box that otherwise repeats the trace name in every tooltip.
- `xaxis=dict(..., tickangle=-45)` - rotates the province labels 45 degrees so they do not collide. This is the standard fix for a crowded categorical axis.

In [ ]:
import plotly.graph_objects as go   # already imported above; harmless to repeat

fig = go.Figure()   # empty container again

fig.add_trace(go.Bar(               # go.Bar instead of go.Scatter: x is categorical
    x=weather['Shortnam'],          # category per bar
    y=weather['temp_diff'],         # bar height
    marker=dict(
        color=weather['temp_diff'], # color by the plotted value itself
        colorscale='RdBu',          # DIVERGING palette: right choice for +/- changes
        line=dict(width=1, color='black')   # bar outline
    ),
    text=weather['temp_diff'].round(1),     # number printed on each bar
    textposition='auto',            # plotly picks inside or outside the bar
    # <extra></extra> removes the gray side box that repeats the trace name
    hovertemplate="<b>%{x}</b><br>Temperature Diff: %{y} F<extra></extra>",
    name='Temp Difference'
))

fig.update_layout(
    title='Temperature Difference by Province (2011 vs 2010)',
    xaxis_title='Province',
    yaxis_title='Temperature Difference (F)',
    width=800,
    height=500,
    plot_bgcolor='white',
    # tickangle=-45 rotates the province labels so they do not overlap
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True, tickangle=-45),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True, gridcolor='lightgrey', zeroline=False),
    font=dict(size=16),
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.show()

### Reshaping before plotting

The plotting call here is one line. The four lines above it are data reshaping, and that is the part worth studying.

1. `weather.sort_values(by='temp_2010', ascending=False).head(3)` - sort warmest-first on the 2010 Fahrenheit column and keep the top 3. Those are NL (28.4 F), MB (26.6 F) and BC (24.8 F). `weather.nlargest(3, 'temp_2010')` does the same in one call.

2. `top_3.set_index('Shortnam')` - promotes the province code from a regular column to the row labels. The index is what pandas uses for alignment, and, importantly here, what `.plot()` uses for naming.

3. `top_3[['temp_2010', 'temp_2011']]` - keep only the two numeric year columns. Anything else would end up in the chart.

4. `top_3.T` - **transpose**: flip rows and columns. Before the flip there are 3 rows (NL, MB, BC) and 2 columns (the two years). After it there are 2 rows (the years) and 3 columns (the provinces).

Why transpose at all? Because `DataFrame.plot(kind='line')` draws **one line per column** and uses the **index as the x axis**. To get years along the bottom and one line per province, the years must be the index and the provinces must be the columns. That is exactly what the transpose produces. Plotting without transposing would give two lines (one per year) against province names on the x axis - a different and less useful chart.

The x axis therefore reads `temp_2010`, `temp_2011`: those are the column names carried across by the transpose, not real dates. The plotly version below fixes this cosmetically with `top_3.index = ['2010', '2011']`.

`top_3.plot(kind='line')` draws it, and pandas builds the legend from the column names automatically. `kind=` also accepts `'bar'`, `'barh'`, `'scatter'`, `'hist'`, `'box'`, `'area'` and `'pie'`.

`plt.xlabel`, `plt.ylabel` and `plt.title` still work afterwards, because `.plot()` drew onto the current matplotlib figure.

In [ ]:
# Sort warmest-first on 2010 and keep 3 rows: NL (28.4 F), MB (26.6 F), BC (24.8 F).
# One-call equivalent: weather.nlargest(3, 'temp_2010')
top_3 = weather.sort_values(by='temp_2010', ascending=False).head(3)

# Temperature over time for the top 3 provinces with provinces as labels and years as x-axis
top_3 = top_3.set_index('Shortnam')          # province code becomes the row label
top_3 = top_3[['temp_2010', 'temp_2011']]    # keep only the two numeric year columns
top_3 = top_3.T                              # TRANSPOSE: rows <-> columns.
                                             # Now 2 rows (years) x 3 columns (provinces).

# .plot(kind='line') draws ONE LINE PER COLUMN and uses the INDEX as the x axis.
# That is why the transpose was needed: years on x, one line per province.
top_3.plot(kind='line')
plt.xlabel('Year')
plt.ylabel('Temperature (F)')
plt.title('Temperature over Time for Top 3 Provinces')
plt.show()

### The same lines in plotly, built with a loop

The reshaping is identical to the previous cell, plus one extra line: `top_3.index = ['2010', '2011']` replaces the row labels so the x axis reads as years rather than as column names. Assigning to `.index` sets every label at once, so the list must be exactly as long as the index.

The new pattern here is the loop:

```python
for province in top_3.columns:
    fig.add_trace(go.Scatter(...))
```

After the transpose, `top_3.columns` is `['NL', 'MB', 'BC']`. The loop runs three times and adds three traces to the same figure, one line per province. This is the standard plotly idiom: **one trace per series**, added in a loop. Plotly gives each trace a different default color and lists all three in the legend.

Inside each `go.Scatter`:

- `x=top_3.index` - the two year labels, shared by all three lines.
- `y=top_3[province]` - that province's column, two values.
- `mode='lines+markers+text'` - draw the connecting line, the two dots, and a text label.
- `name=province` - the legend entry.
- `text=[province, province]` - a label at each of the two points. Plotly needs one entry per point, hence the name twice.
- `textposition='top center'` - where each label sits relative to its dot.
- `line=dict(width=3)` - this `line` is the connecting line itself. It sits at the top level of the trace, not inside `marker`, so it is not a dot border.
- `marker=dict(size=12)` - dot size.

`legend_title="Province"` inside `update_layout` puts a heading above the legend.

Plotly express equivalent: `px.line(top_3, markers=True)` reads a wide DataFrame directly and creates one line per column with no loop at all.

In [ ]:
import plotly.graph_objects as go

# Sort and reshape for Plotly (as you did)
top_3 = weather.sort_values(by='temp_2010', ascending=False).head(3)
top_3 = top_3.set_index('Shortnam')
top_3 = top_3[['temp_2010', 'temp_2011']]
top_3 = top_3.T
top_3.index = ['2010', '2011']  # Label the years
# Assigning to .index replaces ALL row labels at once, so the list length must
# match the index length (2 here). This is purely cosmetic: it fixes the x axis.

fig = go.Figure()

# ONE TRACE PER SERIES. After the transpose, top_3.columns is ['NL', 'MB', 'BC'],
# so this loop runs three times and adds three lines to the same figure.
for province in top_3.columns:
    fig.add_trace(go.Scatter(
        x=top_3.index,              # the two year labels, shared by all lines
        y=top_3[province],          # that province's two temperature values
        mode='lines+markers+text',  # connecting line + dots + labels
        name=province,              # legend entry
        text=[province, province],  # one label per point, so the name twice
        textposition='top center',  # label placement relative to each dot
        line=dict(width=3),         # the CONNECTING line (not inside marker)
        marker=dict(size=12)        # dot size
    ))

fig.update_layout(
    title='Temperature over Time for Top 3 Provinces',
    xaxis_title='Year',
    yaxis_title='Temperature (F)',
    width=800,
    height=500,
    plot_bgcolor='white',
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True, gridcolor='lightgrey'),
    font=dict(size=16),
    margin=dict(l=60, r=40, t=80, b=60),
    legend_title="Province"         # heading above the legend
)

fig.show()

### Fitting a straight line, and reading the regression table

The next cell plots the raw scatter, fits a line, redraws with the line on top, then prints a proper regression table. Take the pieces in order.

**A mismatch with the task text, so you are not confused.** Item 3 of the Task list says "for the years 2010 and 2011", but this cell regresses the original `Temperature` and `Precipitation` columns from the CSV, not `temp_2010` and `temp_2011`. Read the code, not the heading. Redoing it with the year columns is a good exercise.

**The subset.** `regression = weather[['Temperature', 'Precipitation']]` keeps the two numeric columns. Double brackets again, so the result is a DataFrame.

**The fit.** `m, b = np.polyfit(x, y, 1)` fits a polynomial of degree `1`, that is, a straight line, by least squares. It returns the coefficients highest power first, so the two returned numbers unpack into slope `m` and intercept `b`. Here `m = 0.102` and `b = 34.442`. Degree `2` would fit a parabola and return three numbers instead of two.

**Drawing the line.** `m*x + b` computes the predicted precipitation at every observed temperature, and `plt.plot(x, m*x + b, color='red')` connects those predictions. `plt.plot` draws lines and `plt.scatter` draws dots; calling both before a single `plt.show()` puts them on the same axes.

**The regression table.** The last three lines redo the same fit with statsmodels, which adds inference on top of the point estimates.

- `X = sm.add_constant(x)` builds the **design matrix**: a table with one row per observation and one column per parameter to be estimated. `add_constant` prepends a column named `const` that is 1.0 in every row.
- **Why a column of ones?** OLS estimates `y = b0*1 + b1*Temperature`. The intercept `b0` is simply the coefficient on a variable that always equals 1. Without that column, statsmodels fits `y = b1*Temperature` and forces the line through the origin, which here would assert zero precipitation at 0 C. `sm.OLS` does **not** add it for you. This is the single most common statsmodels mistake, and it happens because R's `lm()` and scikit-learn's `LinearRegression` both add the intercept automatically.
- `sm.OLS(y, X)` sets up Ordinary Least Squares: pick the coefficients that minimise the sum of squared vertical distances between the observed `y` values and the fitted line. Watch the argument order - **outcome first, regressors second**. That is the reverse of scikit-learn's `fit(X, y)`.
- `.fit()` runs the estimation and returns a results object. `model.summary()` formats it as the familiar table.

**Reading this particular summary.**

- `No. Observations: 13`, `Df Model: 1`, `Df Residuals: 11` - 13 rows, one slope estimated, and 13 - 2 = 11 residual degrees of freedom.
- `coef` for `Temperature` is `0.1020`: one degree C warmer is associated with 0.102 mm more precipitation on average, in this sample.
- `std err` of `1.190` is the uncertainty attached to that estimate. The coefficient is far smaller than its own standard error, which is a bad sign.
- `t = 0.086` is `coef / std err`. `P>|t| = 0.933` is the p-value: if the true slope were zero, an estimate at least this far from zero would appear 93 percent of the time. Nothing here is distinguishable from noise.
- `[0.025  0.975]` is the 95 percent confidence interval, `-2.517` to `2.721`. It contains zero, which is the same conclusion the p-value gives.
- `R-squared: 0.001` - the fitted line explains about 0.1 percent of the variation in precipitation. With a single regressor, R-squared is the squared correlation, and the correlation here is 0.026.
- `Adj. R-squared: -0.090` - R-squared penalised for the number of regressors. It can go negative when the model is worse than simply predicting the mean of `y` for every observation, and it has gone negative here.
- `Prob (F-statistic): 0.933` - the joint test that all slopes are zero. With one regressor it matches the coefficient p-value exactly.

**The honest conclusion:** in these 13 provincial observations, temperature and precipitation are essentially unrelated. That is a legitimate finding, not a failure. Also note that 13 observations is very few, which is part of why the confidence interval is so wide.

`np.polyfit` and `sm.OLS` return the same slope and intercept, because they solve the same least-squares problem. `polyfit` gives you two numbers. Statsmodels gives you standard errors, p-values, confidence intervals and diagnostics. Use `polyfit` to draw a line, statsmodels to make a claim.

A third option with the same result and a formula syntax closer to R:

```python
import statsmodels.formula.api as smf
smf.ols('Precipitation ~ Temperature', data=weather).fit().summary()
```

The formula interface adds the intercept for you, so no `add_constant` call is needed.

In [ ]:
# Double brackets: a list of names, so this is a DataFrame with two columns.
regression = weather[['Temperature', 'Precipitation']]

# Scatter plot of temperature vs precipitation
plt.scatter(regression['Temperature'], regression['Precipitation'])
plt.xlabel('Temperature (C)')
plt.ylabel('Precipitation (mm)')
plt.title('Temperature vs Precipitation')
plt.show()

# Linear regression
x = regression['Temperature']      # explanatory variable
y = regression['Precipitation']    # outcome variable

# polyfit(x, y, 1) fits a degree-1 polynomial (a straight line) by least squares
# and returns coefficients highest power first, so m = slope, b = intercept.
# Here m = 0.102 and b = 34.442. Degree 2 would return three numbers.
m, b = np.polyfit(x, y, 1)

plt.scatter(x, y)                        # the 13 observed points
plt.plot(x, m*x + b, color='red')        # predicted y at each observed x = the line
plt.xlabel('Temperature (C)')
plt.ylabel('Precipitation (mm)')
plt.title('Temperature vs Precipitation')
plt.show()

# Regression results
# add_constant prepends a column of 1.0s named 'const'. That column IS the
# intercept: OLS fits y = b0*1 + b1*Temperature. statsmodels does NOT add it
# automatically, unlike R's lm() or sklearn, so forgetting this forces the line
# through the origin.
X = sm.add_constant(x)

# sm.OLS(y, X): OUTCOME FIRST, regressors second (the reverse of sklearn's fit(X, y)).
# .fit() runs the estimation; summary() formats the results table.
model = sm.OLS(y, X).fit()
model.summary()

### The fit as an interactive plotly chart

Two traces on one figure, which is the whole point of this cell.

It reuses `x`, `y`, `m` and `b` from the previous cell. Those are ordinary variables still sitting in memory. Run this cell without running that one first and you get `NameError`.

- **Trace 1** is `go.Scatter(..., mode='markers')`: the 13 observed points, drawn as blue dots with a black outline.
- **Trace 2** is `go.Scatter(..., mode='lines')`: the fitted line. `x_sorted = np.sort(x)` sorts the temperatures ascending before computing `y_sorted_fit = m * x_sorted + b`. **That sort matters.** Plotly connects points in the order it receives them, so an unsorted x would draw a zigzag doubling back on itself instead of a clean line. Matplotlib behaves the same way; the previous cell gets away without sorting only because a straight line drawn back over itself still looks straight.
- `hoverinfo='skip'` on the line switches off its tooltip, so hovering reports real data points rather than arbitrary positions along the fit.
- `name=` on each trace is what appears in the legend, and clicking a legend entry hides or shows that trace.

`y_fit = m * x + b` is computed at the top and then never used, because the line trace uses `y_sorted_fit` instead. Harmless, but that is why it looks orphaned.

Plotly express equivalent: `px.scatter(weather, x='Temperature', y='Precipitation', trendline='ols')` fits the regression and draws both layers itself. It calls statsmodels under the hood, which is why statsmodels must be installed for `trendline` to work.

In [ ]:
import plotly.graph_objects as go

# x and y are your original arrays/Series
# Reuses x, y, m and b from the previous cell. Running this alone gives NameError.
y_fit = m * x + b  # regression line using your computed m and b
                   # (computed here but not actually used; the line trace below
                   #  uses y_sorted_fit instead)

fig = go.Figure()

# TRACE 1: the observed data points.
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='markers',                 # dots only, no connecting line
    marker=dict(
        color='blue',               # a single color name, so all dots match
        size=10,
        line=dict(width=1, color='black'),
        opacity=0.8
    ),
    name='Data points',
    hovertemplate='Temperature: %{x} C<br>Precipitation: %{y} mm'
))

# Regression line (sorted for smoothness)
# SORTING MATTERS: plotly connects points in the order given, so unsorted x
# would draw a zigzag doubling back on itself instead of a straight line.
x_sorted = np.sort(x)
y_sorted_fit = m * x_sorted + b

# TRACE 2: the fitted line, added to the SAME figure so both layers overlap.
fig.add_trace(go.Scatter(
    x=x_sorted,
    y=y_sorted_fit,
    mode='lines',                   # line only, no markers
    line=dict(color='red', width=3),
    name='Regression Line',
    hoverinfo='skip'                # no tooltip on the fitted line
))

fig.update_layout(
    title='Temperature vs Precipitation with Regression Line',
    xaxis_title='Temperature (C)',
    yaxis_title='Precipitation (mm)',
    width=800,
    height=500,
    plot_bgcolor='white',
    xaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True),
    yaxis=dict(showline=True, linewidth=2, linecolor='black', mirror=True, gridcolor='lightgrey'),
    font=dict(size=16),
    margin=dict(l=60, r=40, t=80, b=60)
)

fig.show()

# Practice Problem: Regression Analysis

For practice, use the weather dataset to build a regression model that predicts the **2011 temperature** of a province using all other available numerical features (such as precipitation, 2010 temperature, and any other relevant columns in your data).

**Instructions:**

1. Choose `Temperature_2011` as your target variable.
2. Use all other numerical columns in the DataFrame as input (predictor) features.
3. Fit an Ordinary Least Squares (OLS) regression model using the `statsmodels` library.
4. Display the OLS summary table and interpret any two statistics from the summary (such as R-squared, p-values, coefficients, or F-statistic).

*Hint:*
- Use `import statsmodels.api as sm`
- Use `sm.OLS()` to fit your model
- Do not forget to add a constant (intercept) term with `sm.add_constant()`

*Your code should look something like:*
```python
import statsmodels.api as sm

X = weather[[all_numerical_features_except_target]]
y = weather["Temperature_2011"]
X = sm.add_constant(X)
model = sm.OLS(y, X).fit()
print(model.summary())


```
Write your code below and then answer these questions:

- Which variable appears to have the strongest relationship with Temperature_2011?

- What does the R-squared value tell you about your model?

### What the worked code below builds

The practice problem asks you to interpret the output, so the code is supplied. Work through what each step constructs. The two written questions at the end of the cell above are still yours to answer.

- The first cell resets with `weather = df.copy()`, back to the four original columns, and displays all 13 rows.
- The second attaches the `temp_2011` list as `Temperature_2011`. This is the **target**, also called the dependent variable or `y`: the quantity the model tries to explain.
- The third cell builds the two objects every regression needs:
  - `X = weather.drop(columns=["Shortnam", "Region", "Temperature_2011"])` - the **predictors**, also called regressors, features, or the design matrix. The two text columns are dropped because OLS needs numbers, and the target is dropped because using the answer as its own predictor is meaningless. What remains is `Temperature` and `Precipitation`, so `X` is 13 rows by 2 columns.
  - `y = weather["Temperature_2011"]` - single brackets, so a Series of 13 values.
  - Neither line displays anything; both are assignments.
- The next two cells display `X` and `y` so you can see both objects before they enter the model. Look at your design matrix before fitting, every time.

A selection that scales better than listing what to drop:

```python
X = weather.select_dtypes(include='number').drop(columns=['Temperature_2011'])
```

`select_dtypes(include='number')` keeps the numeric columns by type rather than by name, so new text columns never sneak in.

If you did want `Region` as a predictor, you would first turn it into indicator columns with `pd.get_dummies(weather['Region'], drop_first=True)`, which creates one 0/1 column per region minus a reference category. With 13 rows and 13 distinct regions that is hopeless here, but this is what "controlling for region" means mechanically.

In [ ]:
weather = df.copy()   # reset once more to the four original CSV columns
weather

In [ ]:
# temp_2011 is redefined here (same 13 values as in Q6) so this section stands
# on its own. Attached as Temperature_2011, this is the TARGET the model explains.
temp_2011 = [-5, -10, -15, -3, -18, -7, -12, -1, -20, -8, -6, -9, -11]

weather['Temperature_2011'] = temp_2011
weather.head()

In [ ]:
# X = the predictors (the design matrix). Drop the two text columns because OLS
# needs numbers, and drop the target because it cannot predict itself.
# What remains is Temperature and Precipitation: 13 rows x 2 columns.
# Type-based alternative that scales better:
#   X = weather.select_dtypes(include='number').drop(columns=['Temperature_2011'])
X = weather.drop(columns = ["Shortnam", "Region","Temperature_2011"])

# y = the target. Single brackets, so a Series of 13 values.
y = weather["Temperature_2011"]
# Nothing displays: both lines are assignments.

In [ ]:
X   # look at the design matrix before fitting: 13 rows, 2 numeric columns

In [ ]:
y   # the target Series: 13 values, index 0-12, dtype int64

### Adding the constant, fitting, and reading the output

`X = sm.add_constant(X)` prepends the `const` column of ones described earlier, turning the 13 x 2 predictor table into 13 x 3. Re-running the cell is safe: `add_constant` defaults to `has_constant='skip'`, so it will not append a second column of ones.

`sm.OLS(y, X).fit()` estimates the model and `print(model.summary())` prints the table. `print()` is used here rather than letting the cell auto-display, which is why the output appears as plain text rather than the boxed rendering seen earlier. Both contain the same information.

How to read this table:

- **Header block.** `Dep. Variable: Temperature_2011` confirms what is being explained. `No. Observations: 13`. `Df Model: 2` counts the two predictors. `Df Residuals: 10` is 13 rows minus 3 estimated parameters (two slopes plus the intercept).
- **Coefficient block.** One row per parameter: `const`, `Temperature`, `Precipitation`. For each row, `coef` is the estimated effect, `std err` its uncertainty, `t` the ratio of the two, `P>|t|` the p-value for the null hypothesis that the coefficient is zero, and the last two columns the 95 percent confidence interval.
- **Interpreting a coefficient in a multiple regression.** It is the average change in `Temperature_2011` associated with a one-unit increase in that predictor, **holding the other predictors fixed**. Units matter: one degree C for `Temperature`, one millimetre for `Precipitation`. Raw coefficient sizes are therefore not comparable across predictors measured on different scales.
- **Comparing predictors.** To judge which predictor has the stronger *statistical* relationship, compare the p-values, or equivalently the absolute `t` values. Comparing raw coefficient magnitudes across differently scaled variables is the classic error. Standardising the variables first is the other legitimate way to make them comparable.
- **`R-squared`** is the share of variance in `Temperature_2011` explained by the two predictors together. **`Adj. R-squared`** penalises for the number of predictors, and when it turns negative the model is doing worse than predicting the sample mean for every province.
- **`Prob (F-statistic)`** tests all slopes jointly against zero. When it is large, the model as a whole has no explanatory power regardless of what any single coefficient row suggests.

Keep the data in mind while you interpret. `Temperature_2011` is an invented list of integers and `Temperature` / `Precipitation` are the CSV values, so there is no reason for a real relationship to exist. The skill being practised is reading the table correctly, not discovering a finding.

In [ ]:
# add_constant prepends the column of 1.0s that becomes the intercept, turning
# X from 13x2 into 13x3. Safe to re-run: the default has_constant='skip' means
# it will not add a second constant column.
X = sm.add_constant(X)

# OUTCOME FIRST, regressors second. .fit() estimates the coefficients.
model = sm.OLS(y, X).fit()

# print() writes the plain-text table. Without print, the cell would auto-display
# the same summary in its boxed form.
print(model.summary())

## Quick reference

Everything used above, in one place.

**Inspecting a table**

| Task | Code |
| --- | --- |
| Load a CSV | `pd.read_csv('path/to/file.csv')` |
| Structure, dtypes, missing counts | `df.info()` |
| Row and column count | `df.shape` |
| First / last rows | `df.head(n)` / `df.tail(n)` |
| Numeric summary statistics | `df.describe()` |
| Count missing values per column | `df.isna().sum()` |

**Selecting**

| Task | Code |
| --- | --- |
| One column, as a Series | `df['Temperature']` |
| Several columns, as a DataFrame | `df[['Temperature', 'Precipitation']]` |
| Rows matching a condition | `df[df['Temperature'] > -10]` |
| Rows and columns together | `df.loc[df['Temperature'] > -10, ['Region']]` |
| By position | `df.iloc[0]`, `df['Temperature'].iloc[0]` |
| Same filter as a string | `df.query("Region == 'Alberta'")` |
| Match any of several values | `df[df['Region'].isin(['Alberta', 'Quebec'])]` |

**Changing a table**

| Task | Code |
| --- | --- |
| Independent duplicate | `df.copy()` |
| New column from arithmetic | `df['F'] = df['C'] * 9/5 + 32` |
| New column from a function | `df['F'] = df['C'].apply(c_to_f)` |
| New column from a two-way rule | `df['flag'] = np.where(df['C'] > 5, 'High', 'Low')` |
| Remove columns | `df = df.drop(columns=['a', 'b'])` |
| Rename columns | `df = df.rename(columns={'old': 'new'})` |
| Sort by a column | `df.sort_values(by='Region', ascending=False)` |
| Top n by a numeric column | `df.nlargest(3, 'Temperature')` |
| Make a column the row labels | `df.set_index('Shortnam')` |
| Restore 0-based labels | `df.reset_index(drop=True)` |
| Flip rows and columns | `df.T` |
| Drop rows with missing values | `df = df.dropna()` |
| Fill missing values instead | `df['x'] = df['x'].fillna(0)` |

**Plotting**

| Task | Code |
| --- | --- |
| Quick chart from a DataFrame | `df.plot(kind='line')` |
| Scatter | `plt.scatter(x, y)` |
| Bar | `plt.bar(categories, heights)` |
| Line | `plt.plot(x, y)` |
| Text at a data coordinate | `plt.text(x, y, 'label')` |
| Labels and render | `plt.xlabel(...)`, `plt.title(...)`, `plt.show()` |
| Interactive, minimal code | `px.scatter(df, x='a', y='b', text='name')` |
| Interactive, full control | `go.Figure()` + `fig.add_trace(...)` + `fig.update_layout(...)` + `fig.show()` |

**Regression**

| Task | Code |
| --- | --- |
| Slope and intercept only | `m, b = np.polyfit(x, y, 1)` |
| Add the intercept column | `X = sm.add_constant(X)` |
| Fit OLS (outcome first) | `model = sm.OLS(y, X).fit()` |
| Full results table | `model.summary()` |
| Coefficients only | `model.params` |
| R-squared only | `model.rsquared` |
| Formula syntax, intercept automatic | `smf.ols('y ~ x1 + x2', data=df).fit()` |

**The five mistakes that cost the most time**

1. Using `=` where you meant `==` inside a filter. It overwrites the column.
2. Using `and` / `or` instead of `&` / `|`, or forgetting the parentheses around each condition.
3. Forgetting to reassign: `df.drop(...)`, `df.rename(...)`, `df.sort_values(...)` and `df.dropna()` all return new objects and change nothing on their own.
4. Forgetting `sm.add_constant`, which silently forces the regression line through the origin.
5. Using `[i]` on a Series and assuming it means position. It means the label. Use `.iloc[i]` for position.

# NumPy and Constrained Optimization

The rest of this notebook covers two tools you will need in FRE 525 and in any applied
economics work that involves choosing quantities to maximize or minimize something.

1. NumPy, the library for numerical arrays. It is what pandas is built on.
2. scipy.optimize, the library for solving optimization problems numerically.

Nothing here assumes you have seen either before.

## Part 1: NumPy arrays

### What problem does NumPy solve?

A Python list can hold numbers, but it cannot do arithmetic on all of them at once.
If `prices = [10, 20, 30]`, then `prices * 2` does NOT double the numbers. It repeats
the list, giving `[10, 20, 30, 10, 20, 30]`. To double the values you would need a loop.

A NumPy array is a container that looks like a list but behaves like a mathematical
vector. Arithmetic applies to every element at once. This is called vectorization.

Two rules that make arrays different from lists:

1. Every element must be the same type, usually all floats or all integers.
2. Arithmetic is elementwise, so no loop is needed.

Run the cell below to see the difference.

In [ ]:
import numpy as np    # 'np' is the universal short name for numpy

prices_list  = [10, 20, 30]           # an ordinary Python list
prices_array = np.array([10, 20, 30]) # the same numbers as a NumPy array

print("list  * 2 :", prices_list * 2)   # repeats the list, probably not what you want
print("array * 2 :", prices_array * 2)  # doubles each number, which is what you want

### Creating arrays

You will rarely type every number by hand. These constructors cover almost everything.

| Call | What it makes | Example result |
|---|---|---|
| `np.array([1, 2, 3])` | an array from a list you already have | `[1 2 3]` |
| `np.arange(5)` | counts 0, 1, 2, ... up to but NOT including 5 | `[0 1 2 3 4]` |
| `np.zeros(4)` | four zeros, useful as an empty container to fill in | `[0. 0. 0. 0.]` |
| `np.ones(4)` | four ones | `[1. 1. 1. 1.]` |
| `np.full(4, 2.5)` | four copies of 2.5, useful as a starting guess | `[2.5 2.5 2.5 2.5]` |
| `np.linspace(0, 1, 5)` | 5 evenly spaced values from 0 to 1 INCLUSIVE | `[0. 0.25 0.5 0.75 1.]` |

Note the difference between `arange` and `linspace`. `arange` is told the step size and
excludes the endpoint. `linspace` is told how many values you want and includes the
endpoint. Use `arange` for time periods 0, 1, 2, ... and `linspace` when you want a smooth
grid for plotting a curve.

In [ ]:
print("np.array([1,2,3])   :", np.array([1, 2, 3]))
print("np.arange(5)        :", np.arange(5))        # 0 to 4, stop value excluded
print("np.arange(2, 10, 3) :", np.arange(2, 10, 3)) # start 2, stop before 10, step 3
print("np.zeros(4)         :", np.zeros(4))
print("np.ones(4)          :", np.ones(4))
print("np.full(4, 2.5)     :", np.full(4, 2.5))
print("np.linspace(0,1,5)  :", np.linspace(0, 1, 5))  # endpoint included

### Inspecting an array

Three attributes tell you what you are holding. There are no parentheses after these,
because they are attributes (facts about the object), not methods (actions it performs).

- `.shape` gives the dimensions as a tuple. `(4,)` means a flat array of 4 values.
- `.dtype` gives the element type. `float64` is a decimal number, `int64` a whole number.
- `.ndim` gives the number of dimensions. 1 is a vector, 2 is a matrix.

In [ ]:
q = np.array([3.0, 2.5, 2.0, 1.5])

print("q       :", q)
print("q.shape :", q.shape)   # (4,) means 4 elements in one dimension
print("q.dtype :", q.dtype)   # float64 because we wrote decimal points
print("q.ndim  :", q.ndim)    # 1 dimension, so this is a vector
print("len(q)  :", len(q))    # number of elements

### Vectorized arithmetic

This is the part that matters most for economics. Any arithmetic you write is applied to
every element. You can combine an array with a single number, or two arrays of the same
length elementwise.

Compare the two blocks below. They give the same answer, but the vectorized version is one
line, is faster, and reads much closer to the algebra you would write on paper.

In [ ]:
q = np.array([3.0, 2.5, 2.0, 1.5])
alpha, gamma = 30.0, 3.0

# The long way, with a loop:
benefit_loop = []
for value in q:
    benefit_loop.append(alpha * value - (gamma / 2) * value**2)
print("loop       :", np.round(benefit_loop, 4))

# The vectorized way, one line, no loop:
benefit_vec = alpha * q - (gamma / 2) * q**2
print("vectorized :", np.round(benefit_vec, 4))

### Powers and discount factors

`q**2` squares every element. The same works with a single base and an array exponent,
which is exactly how you build a series of discount factors.

In economics, a payoff received `t` periods from now is worth `beta**t` times its face
value today, where `beta = 1 / (1 + r)` and `r` is the discount rate. Put the time periods
in an array with `np.arange(T)` and `beta**t` gives the whole sequence in one step.

In [ ]:
r    = 0.08            # discount rate, 8 percent per period
beta = 1 / (1 + r)     # discount factor
T    = 4               # number of periods

t = np.arange(T)       # time periods: [0 1 2 3]
discount = beta**t     # discount factor for each period

print("t             :", t)
print("beta          :", round(beta, 6))
print("beta**t       :", np.round(discount, 6))
print("check period 2:", round(beta**2, 6), "which matches discount[2]")

### Summarizing an array

These reduce a whole array to a single number. They are methods, so they take parentheses.

| Method | Returns |
|---|---|
| `q.sum()` | total of all elements |
| `q.mean()` | arithmetic average |
| `q.min()`, `q.max()` | smallest and largest value |
| `q.argmax()` | the POSITION of the largest value, not the value itself |
| `np.abs(q)` | absolute value of each element |

`argmax` is the one people misread. It answers "which period was the largest?", not
"how large was it?".

In [ ]:
q = np.array([3.0, 2.5, 4.0, 1.5])

print("sum    :", q.sum())
print("mean   :", q.mean())
print("max    :", q.max())      # the largest VALUE
print("argmax :", q.argmax())   # the POSITION of the largest value, counting from 0
print("so the largest value is q[", q.argmax(), "] =", q[q.argmax()])

### Boolean masks and filtering

Comparing an array to a number gives an array of True and False, one per element. That is
a boolean mask. Putting the mask inside square brackets keeps only the elements where the
mask is True.

This is the same idea as the filtering you did earlier on the weather table, where
`weather[weather['Region'] == 'Alberta']` kept only the rows whose mask was True.
pandas borrowed the idea from NumPy.

In [ ]:
q = np.array([3.0, 2.5, 4.0, 1.5])

mask = q > 2.0
print("q            :", q)
print("q > 2.0      :", mask)        # one True or False per element
print("q[q > 2.0]   :", q[mask])     # keeps only elements where the mask is True
print("how many     :", mask.sum())  # True counts as 1, so sum() counts the matches

### Indexing and slicing an array

Indexing works exactly like a Python list, and like the string indexing in the prep package.

- Counting starts at 0, so `q[0]` is the FIRST element.
- `q[-1]` is the last element, `q[-2]` the second to last.
- `q[1:3]` is a slice from position 1 up to but NOT including position 3.
- Leaving a side blank means "all the way to that end", so `q[:2]` is the first two and
  `q[2:]` is everything from position 2 onward.

The excluded stop value is the most common source of off by one errors. `q[1:3]` gives you
two elements, not three.

In [ ]:
q = np.array([3.0, 2.5, 4.0, 1.5])

print("q       :", q)
print("q[0]    :", q[0])     # first element
print("q[-1]   :", q[-1])    # last element
print("q[1:3]  :", q[1:3])   # positions 1 and 2, position 3 is excluded
print("q[:2]   :", q[:2])    # first two
print("q[2:]   :", q[2:])    # from position 2 to the end

### Which container should you use?

| Use | When |
|---|---|
| Python list | a handful of mixed things, no maths needed |
| NumPy array | numbers of one type, you want arithmetic, and there are no column names |
| pandas DataFrame | a table with named columns and mixed types, which is most real data |

A pandas column is a NumPy array underneath. `df['Temperature'].values` hands you the raw
array. The two libraries are partners: pandas for loading, cleaning and labelling, NumPy
for the arithmetic.

## Part 2: scipy.optimize

### What an optimization problem is

Three ingredients:

1. Decision variables: the numbers you get to choose. Below, how much water to apply in
   each month.
2. Objective function: a single number you want as large or as small as possible.
3. Constraints: rules the choice must obey, for example a fixed total budget, or a
   requirement that quantities cannot be negative.

`scipy.optimize.minimize` searches for the decision variables that make the objective as
small as possible while respecting the constraints.

It only ever minimizes. To maximize something, minimize its negative. That is why the
function below is called `neg_pv` and ends with a minus sign. This trips up almost
everyone the first time.

### The smallest possible example

Before the economics, look at the mechanics on a function whose answer you already know.
`f(x) = (x - 3)**2 + 2` is a parabola with its lowest point at `x = 3`, where `f = 2`.

`minimize` needs two things at minimum: the function, and a starting guess. It then walks
downhill from that guess until it cannot improve.

In [ ]:
from scipy.optimize import minimize

def f(x):
    return (x - 3)**2 + 2       # lowest point is at x = 3, where f = 2

result = minimize(f, x0=[0.0])  # x0 is the starting guess, given as a list

print("success :", result.success)                # did it converge?
print("x*      :", round(float(result.x[0]), 6))  # the minimizing x, should be 3
print("f(x*)   :", round(float(result.fun), 6))   # the minimum value, should be 2

### Reading the result object

`minimize` returns one object holding everything about the search. The four fields you will
actually use:

| Field | Meaning |
|---|---|
| `result.x` | the optimal decision variables, always returned as an array |
| `result.fun` | the value of the objective AT the optimum |
| `result.success` | True if it converged, False if it gave up |
| `result.message` | plain text explaining why it stopped |

Always check `result.success`. If it is False, the numbers in `result.x` are not a solution
and must not be reported. And if you minimized a negative in order to maximize, the true
maximum is `-result.fun`, not `result.fun`.

### Bounds and constraints

Bounds restrict each variable individually to a range. Pass a list with one `(low, high)`
pair per variable, using `None` for no limit. To force four variables to be non-negative:
`bounds = [(0, None)] * 4`.

Constraints are rules that link variables together, written as a dictionary:

```python
{'type': 'ineq', 'fun': lambda q: S0 - q.sum()}
```

- `'type': 'ineq'` means scipy will force `fun(q) >= 0`. Here that is `S0 - q.sum() >= 0`,
  which says the total used cannot exceed the budget `S0`.
- `'type': 'eq'` would instead force `fun(q) == 0`, an exact equality.
- `lambda q: ...` is a short way of writing a one line function without naming it.
  `lambda q: S0 - q.sum()` means the same as `def constraint(q): return S0 - q.sum()`.

You must write the rule so the acceptable region is where the expression is zero or
positive. Writing `q.sum() - S0` instead would enforce the opposite and give nonsense.

Method: `method='SLSQP'` stands for Sequential Least Squares Quadratic Programming. It is
the standard choice here because it is one of the few methods that handles bounds AND
constraints together. Methods such as BFGS ignore constraints entirely.

### Worked example: allocating a fixed water budget

A farm has a fixed irrigation allocation of `S0 = 12` units to spread over a `T = 4` month
growing season. It cannot buy more.

Applying `q` units in a month produces benefit

    W(q) = alpha * q - (gamma / 2) * q**2,   with alpha = 30 and gamma = 3

The squared term makes the benefit diminishing: the first unit of water is worth far more
than the tenth. Benefit received `t` months from now is discounted by `beta**t` where
`beta = 1 / (1 + r)` and `r = 0.08`.

The farm chooses `q_0, q_1, q_2, q_3` to maximize total discounted benefit, subject to
using no more than 12 units in total and never applying a negative amount:

    maximize    sum over t of  beta**t * (alpha * q_t - (gamma/2) * q_t**2)
    subject to  q_0 + q_1 + q_2 + q_3  <=  12
                q_t >= 0 for every t

Now translate each line of that into code.

In [ ]:
import numpy as np
from scipy.optimize import minimize

# ---- 1. Parameters ------------------------------------------------------
alpha, gamma = 30.0, 3.0   # benefit curve: W(q) = alpha*q - (gamma/2)*q^2
S0 = 12.0                  # total water available for the whole season
r  = 0.08                  # discount rate per month
T  = 4                     # number of months
beta = 1 / (1 + r)         # discount factor
t = np.arange(T)           # month numbers [0 1 2 3]

# ---- 2. Objective function ----------------------------------------------
# minimize() only minimizes, so return the NEGATIVE of what we want to maximize.
def neg_pv(q):
    W = alpha * q - (gamma / 2) * q**2   # benefit each month, vectorized over q
    return -(beta**t * W).sum()          # discount, add up, then negate

# ---- 3. Bounds: every month's water must be zero or positive ------------
bounds = [(0, None)] * T

# ---- 4. Constraint: total used cannot exceed the allocation -------------
# 'ineq' enforces fun(q) >= 0, i.e. S0 - sum(q) >= 0, i.e. sum(q) <= S0
budget = {'type': 'ineq', 'fun': lambda q: S0 - q.sum()}

# ---- 5. Starting guess: split the water evenly across the months --------
q_start = np.full(T, S0 / T)

# ---- 6. Solve -----------------------------------------------------------
res = minimize(neg_pv, q_start, method='SLSQP', bounds=bounds,
               constraints=budget, options={'ftol': 1e-12, 'maxiter': 1000})

q_opt = res.x
pv_opt = -res.fun          # undo the negation to get the true maximum

print("converged      :", res.success)
print("optimal q      :", np.round(q_opt, 4))
print("total water    :", round(q_opt.sum(), 6), "out of", S0)
print("max discounted :", round(pv_opt, 4))

### Reading the answer

Three things to check every time, before believing any optimization result.

1. Did it converge? `res.success` must be True.

2. Is the constraint binding? Total water used comes out at exactly 12, the whole
   allocation. That makes economic sense: benefit is still increasing at the optimum, so
   there is no reason to leave water unused.

3. Does the pattern make economic sense? The optimal amounts fall over time, roughly
   3.79, 3.29, 2.75, then 2.17. Water is applied more heavily early. That is the
   discounting at work: a unit of benefit next month is worth less than one this month, so
   the farm front loads.

There is a sharper way to state the same result. At the optimum the discounted marginal
benefit must be equal in every month. If it were higher in one month you could move a unit
of water there and do better, so you would not be at the optimum. Marginal benefit here is

    MB(q) = alpha - gamma * q

The cell below checks that `beta**t * MB(q_t)` is the same number in every month, and that
the undiscounted marginal benefit therefore grows at exactly the rate `1 + r`.

This growth-at-the-interest-rate result is the same logic behind Hotelling's Rule for
extracting a non-renewable resource, which you will meet in FRE 525.

In [ ]:
mb = alpha - gamma * q_opt          # marginal benefit in each month

print("marginal benefit           :", np.round(mb, 4))
print("discounted marginal benefit:", np.round(beta**t * mb, 6))
print("  -> equal across months, which is the efficiency condition")
print()
print("ratio MB[t+1] / MB[t]      :", np.round(mb[1:] / mb[:-1], 6))
print("  -> equals 1 + r =", 1 + r, "so marginal benefit grows at the discount rate")

### Things that go wrong

| Symptom | Cause | Fix |
|---|---|---|
| You get a minimum when you wanted a maximum | forgot to negate | return `-value` from the objective, and report `-res.fun` |
| `res.success` is False | bad starting guess, or constraints cannot all be satisfied | try a different `x0`, check the constraint signs |
| Constraint appears ignored | wrong sign, or a method that does not support constraints | for `ineq` the feasible side must be `>= 0`; use `method='SLSQP'` |
| Answer changes between runs | flat objective, or tolerance too loose | tighten with `options={'ftol': 1e-12}` |
| Negative quantities in the answer | no bounds given | pass `bounds=[(0, None)] * T` |

### Alternatives

- `minimize_scalar` when there is exactly one decision variable and no constraints. Simpler,
  and needs no starting guess.
- Other `method=` choices: `'BFGS'` is fast for smooth unconstrained problems but ignores
  constraints. `'Nelder-Mead'` needs no derivatives and copes with rough functions, but is
  slow and also ignores constraints. `'SLSQP'` and `'trust-constr'` handle constraints
  properly.
- Solve it by hand. This problem has a closed form, because setting discounted marginal
  benefits equal gives a small system you can solve with algebra. When a closed form exists
  it is exact and instant. Use the numerical solver to CHECK your algebra, and rely on it
  when the problem is too messy to solve on paper.